# Exploratory Data Analysis: Zomato Restaurants
**Dataset:** Zomato Restaurants dataset from Kaggle (9,551 restaurants, 21 columns)  
**Questions I want to answer:** What drives a restaurant's rating? Do delivery, table booking, price and cuisine matter?

> Replace every `TODO` with your own observation after looking at the output above it.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

df = pd.read_csv("Zomato_Restaurant_Dataset.csv")

## 1. Structural overview

In [ ]:
print("Shape:", df.shape)
print(f"Memory usage: {df.memory_usage(deep=True).sum()/1e6:.2f} MB")
pd.DataFrame({"dtype": df.dtypes, "unique": df.nunique()})

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

**Notes on the data (to check as you go):**
- `Aggregate rating = 0` appears to mean *not rated*, not a terrible score. 
- `Currency` is mixed, so `Average Cost for two` can't be compared across countries without filtering.
- `Switch to order menu` has a single value, so it carries no information.

## 2. Missing values

In [ ]:
miss = df.isna().sum()
miss_df = pd.DataFrame({"missing": miss, "pct": (miss/len(df)*100).round(3)}).query("missing > 0")
miss_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(df.isna().sum()/len(df)*100).sort_values().plot.barh(ax=ax[0], color="tomato")
ax[0].set_title("Percentage of missing values per column")
ax[0].set_xlabel("% missing"); ax[0].set_ylabel("Column")
sns.heatmap(df.isna(), cbar=False, yticklabels=False, ax=ax[1])
ax[1].set_title("Missing-value map (light = missing)")
ax[1].set_xlabel("Column"); ax[1].set_ylabel("Row")
plt.tight_layout(); plt.show()

**Interpretation:** TODO (which column has missing data, how much, and is it a problem?)

## 3. Summary statistics

In [ ]:
stats = df[num_cols].describe().T
stats["median"] = df[num_cols].median()
stats[["mean","median","std","min","25%","50%","75%","max"]].round(2)

In [ ]:
for c in ["Has Table booking","Has Online delivery","Is delivering now","Rating text","Currency","City"]:
    print(f"--- {c} ---")
    print(df[c].value_counts().head(8), "\n")

In [ ]:
print((df["Aggregate rating"] == 0).sum(), "restaurants have rating 0 (Not rated)")
print(df.loc[df["Aggregate rating"]==0, "Rating text"].unique())

**Interpretation:** TODO (anything surprising: skew in cost/votes, imbalance in yes/no columns, geographic concentration?)

## 4. Data preparation
Prepared views used for the charts below:
- `rated`: only restaurants with a real rating (> 0)
- `india`: Indian restaurants in rupees, so cost is comparable

In [ ]:
rated = df[df["Aggregate rating"] > 0].copy()
india = df[df["Currency"].str.contains("Rs.", regex=False)].copy()
india_rated = india[india["Aggregate rating"] > 0]
print(len(rated), "rated restaurants;", len(india), "Indian restaurants")

## 5. Visualisations

### Q1. How are restaurant ratings distributed?

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(rated["Aggregate rating"], bins=25, kde=True, ax=ax)
ax.axvline(rated["Aggregate rating"].median(), color="red", ls="--", label="median")
ax.set_title("Distribution of aggregate ratings (rated restaurants only)")
ax.set_xlabel("Aggregate rating"); ax.set_ylabel("Number of restaurants"); ax.legend()
plt.show()

**Interpretation:** TODO

### Q2. Which cuisines are most common, and which are rated highest?

In [ ]:
cu = rated.dropna(subset=["Cuisines"]).assign(Cuisine=lambda d: d["Cuisines"].str.split(", ")).explode("Cuisine")
g = cu.groupby("Cuisine")["Aggregate rating"].agg(["count","mean"])
top = g.sort_values("count", ascending=False).head(12)
fig, ax = plt.subplots(1, 2, figsize=(14,5))
top["count"].sort_values().plot.barh(ax=ax[0], color="steelblue")
ax[0].set_title("12 most common cuisines"); ax[0].set_xlabel("Number of restaurants"); ax[0].set_ylabel("Cuisine")
top["mean"].sort_values().plot.barh(ax=ax[1], color="seagreen")
ax[1].set_title("Average rating of those cuisines"); ax[1].set_xlabel("Mean rating"); ax[1].set_ylabel("Cuisine")
ax[1].set_xlim(2.5, 4)
plt.tight_layout(); plt.show()

**Interpretation:** TODO (is popularity the same as quality?)

### Q3. Do restaurants with online delivery or table booking get better ratings?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,5), sharey=True)
sns.boxplot(data=rated, x="Has Online delivery", y="Aggregate rating", ax=ax[0])
ax[0].set_title("Rating by online delivery"); ax[0].set_xlabel("Has online delivery"); ax[0].set_ylabel("Aggregate rating")
sns.boxplot(data=rated, x="Has Table booking", y="Aggregate rating", ax=ax[1])
ax[1].set_title("Rating by table booking"); ax[1].set_xlabel("Has table booking"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()
print(rated.groupby("Has Online delivery")["Aggregate rating"].mean().round(2))
print(rated.groupby("Has Table booking")["Aggregate rating"].mean().round(2))

**Interpretation:** TODO (difference in medians? remember correlation is not causation)

### Q4. Do pricier restaurants get higher ratings?

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(data=rated, x="Price range", y="Aggregate rating", ax=ax)
ax.set_title("Rating by price range (1 = cheapest, 4 = most expensive)")
ax.set_xlabel("Price range"); ax.set_ylabel("Aggregate rating")
plt.show()

**Interpretation:** TODO

### Q5. Is a higher rating linked to more votes?

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sns.scatterplot(data=rated, x="Votes", y="Aggregate rating", alpha=.3, ax=ax)
ax.set_xscale("log")
ax.set_title("Votes (log scale) vs aggregate rating")
ax.set_xlabel("Number of votes (log scale)"); ax.set_ylabel("Aggregate rating")
plt.show()
print("Spearman correlation:", round(rated["Votes"].corr(rated["Aggregate rating"], method="spearman"), 2))

**Interpretation:** TODO

### Q6. Which cities have the most restaurants, and how does average cost compare (India only)?

In [ ]:
topc = india["City"].value_counts().head(8).index
fig, ax = plt.subplots(1, 2, figsize=(14,5))
india["City"].value_counts().head(8).sort_values().plot.barh(ax=ax[0], color="steelblue")
ax[0].set_title("Top 8 Indian cities by number of restaurants"); ax[0].set_xlabel("Number of restaurants"); ax[0].set_ylabel("City")
sns.boxplot(data=india[india["City"].isin(topc) & (india["Average Cost for two"] < 5000)],
            y="City", x="Average Cost for two", ax=ax[1])
ax[1].set_title("Average cost for two (Rs., under 5000)"); ax[1].set_xlabel("Average cost for two (Rs.)"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

**Interpretation:** TODO (note how concentrated the dataset is in the Delhi NCR area)

### Q7. Which numeric variables are correlated?

In [ ]:
cols = ["Average Cost for two","Price range","Aggregate rating","Votes"]
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(india_rated[cols].corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwith" if False else "coolwarm", center=0, ax=ax)
ax.set_title("Spearman correlation (rated Indian restaurants)")
plt.show()

**Interpretation:** TODO

## 6. Key findings
1. TODO
2. TODO
3. TODO
4. TODO

**Limitations:** the dataset is dominated by Delhi NCR; ~22% of restaurants are unrated; currencies are mixed.